# Figure 4: kappa vs mean degree for beta-binomial degrees, constant threshold

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import poisson, betabinom, beta as beta_dist
from figstyle import PALETTE, LW, REF_LW, FIGW_SINGLE, FIGW_DOUBLE, save

In [ ]:
# Figure 4: kappa vs mean degree for beta-binomial degree x constant threshold,
# for a sweep of alpha_d, with the ER (Poisson) reference.
phi = 0.25
k_trig = int(1 / phi)
n_max = 5000
n_vals = np.arange(2, n_max + 1)
beta_bb = 20.0

alpha_sets = [
    (4.,   r'$\alpha_d=4$'),
    (3.,   r'$\alpha_d=3$'),
    (2.,   r'$\alpha_d=2$'),
    (1.,   r'$\alpha_d=1$'),
    (0.5,  r'$\alpha_d=0.5$'),
    (0.25, r'$\alpha_d=0.25$'),
]

fig, ax = plt.subplots(figsize=(FIGW_SINGLE, 2.7))

# ER (Poisson) reference curve
z_ref = np.linspace(0.05, 8, 800)
kappa_er = []
for z in z_ref:
    pk = poisson.pmf(np.arange(k_trig + 1), z)
    pk[0] = 0.0
    ks = np.arange(k_trig + 1)
    kappa_er.append(np.sum(ks[2:] * (ks[2:] - 1) * pk[2:]) / z)
ax.plot(z_ref, kappa_er, lw=REF_LW, color='black', ls='--', label='ER (Poisson)')

# Beta-binomial curves, one per alpha_d
ks = np.arange(n_max + 1)
results = {}
for i, (alpha_d, label) in enumerate(alpha_sets):
    kappas, means = [], []
    for n in n_vals:
        pk = betabinom.pmf(ks[:n + 1], n, alpha_d, beta_bb)
        mk = (ks[:len(pk)] * pk).sum()
        if mk < 1e-10:
            continue
        kmin = min(k_trig + 1, len(pk))
        w = ks[2:kmin] * (ks[2:kmin] - 1) * pk[2:kmin] / mk
        kappas.append(w.sum())
        means.append(mk)
    results[alpha_d] = (np.array(means), np.array(kappas))
    ax.plot(means, kappas, lw=LW, color=PALETTE[i], label=label)

ax.axhline(1.0, color='gray', ls=':', lw=REF_LW, alpha=0.6)
ax.set_xlabel(r'$\langle k \rangle$')
ax.set_ylabel(r'$\kappa$')
ax.set_xlim(0, 8)
ax.legend(loc='upper right', frameon=False)
fig.tight_layout()
save(fig, "bb-constant")
plt.show()

In [ ]:
# checks: the kappa>1 window narrows as alpha_d decreases and vanishes for small alpha_d
def upper_z(ad):
    m, k = results[ad]
    sel = (k > 1) & (m <= 8)
    return m[sel].max() if sel.any() else 0.0

order = [4., 3., 2., 1., 0.5, 0.25]
us = [upper_z(ad) for ad in order]
for ad, u in zip(order, us):
    print(f"alpha_d={ad:g}: kappa>1 for z <= {u:.2f}")
assert all(us[i] >= us[i + 1] for i in range(len(us) - 1)), "window should narrow with alpha_d"
assert us[-1] == 0.0 and us[-2] == 0.0, "window should vanish for small alpha_d"
print("figure 4 checks passed")